# Week 9 - Quora Question Pairs: Problem Definition & Data Overview

**Objective:** Detect whether two questions asked on Quora have the **same intent** (duplicates). Over the next 4 weeks: EDA, TF-IDF features, classical ML models and performance measures (precision, recall, F1, accuracy, log-loss).

## a. Business / real-world problem
On a question-answering platform, many users ask the same thing in different words:
*"How do I learn Python?"* vs *"What's the best way to start with Python?"*.
Detecting duplicates lets the platform merge them, keep answers in one place, improve search, and send users straight to existing answers.

## b. Business objectives and constraints
* Detect duplicate pairs **accurately**.
* Reduce repeated questions.
* **Minimise false positives** (different questions wrongly merged - bad: a user's real question gets hidden) and **false negatives** (duplicates missed).
* Provide a **probability** of being a duplicate, so the decision threshold can be tuned.
* Handle **large text volume** within reasonable memory and time (400k pairs here).

## c. Mapping to an ML problem
* **Binary classification** with input `(question1, question2)` and target `is_duplicate` (1 = same intent, 0 = different).
* **Data:** the Kaggle *Quora Question Pairs* dataset - 404,290 labelled pairs. Columns: `id`, `qid1`, `qid2`, `question1`, `question2`, `is_duplicate`.

## Getting `train.csv`
Kaggle's `train.csv` needs a Kaggle login to download. If `data/quora/train.csv` isn't present, the cell below rebuilds an equivalent file from **GLUE QQP**, which is the same Quora data (≈404k pairs) published on Hugging Face. If you have the original Kaggle file, just put it at `data/quora/train.csv` and this cell does nothing.

Differences vs Kaggle's file: GLUE doesn't ship Quora's question ids, so `qid1`/`qid2` are regenerated (same text -> same id) and the row count is 404,276 instead of 404,290 (GLUE removed a few malformed rows).

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import os
os.makedirs("data/quora", exist_ok=True)

if not os.path.exists("data/quora/train.csv"):
    base = "https://huggingface.co/datasets/nyu-mll/glue/resolve/main/qqp/"
    parts = [pd.read_parquet(base + f"{s}-00000-of-00001.parquet") for s in ("train", "validation")]
    raw = pd.concat(parts, ignore_index=True)
    ids = {q: i + 1 for i, q in enumerate(pd.unique(pd.concat([raw["question1"], raw["question2"]])))}
    out = pd.DataFrame({
        "id": np.arange(len(raw)),
        "qid1": raw["question1"].map(ids), "qid2": raw["question2"].map(ids),
        "question1": raw["question1"], "question2": raw["question2"],
        "is_duplicate": raw["label"],
    })
    out.to_csv("data/quora/train.csv", index=False)
    print("train.csv created")
else:
    print("train.csv already present")

train.csv created


## c. Load and inspect the dataset

In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/quora/train.csv")

In [3]:
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df[["question1", "question2", "is_duplicate"]].isnull().sum())

display(df.head())
print("\nTarget distribution:")
print(df["is_duplicate"].value_counts())

Dataset shape: (404276, 6)

Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']

Missing values:
 question1       0
question2       0
is_duplicate    0
dtype: int64


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,290449,How is the life of a math student? Could you d...,Which level of prepration is enough for the ex...,0
1,1,2,290450,How do I control my horny emotions?,How do you control your horniness?,1
2,2,3,290451,What causes stool color to change to yellow?,What can cause stool to come out as little balls?,0
3,3,4,290452,What can one do after MBBS?,What do i do after my MBBS ?,1
4,4,5,290453,Where can I find a power outlet for my laptop ...,"Would a second airport in Sydney, Australia be...",0



Target distribution:
is_duplicate
0    255013
1    149263
Name: count, dtype: int64


**Reading it:** about 63% of pairs are *not* duplicates and 37% are. This **class imbalance** means a model that always says "not duplicate" would score ~63% accuracy while being useless - this is why we also track precision, recall, F1 and log-loss (Week 10).

A few example pairs, to get a feel for the task:

In [4]:
for label in (1, 0):
    print("=== is_duplicate =", label)
    for _, r in df[df["is_duplicate"] == label].sample(3, random_state=1).iterrows():
        print("  Q1:", r["question1"]); print("  Q2:", r["question2"]); print()

=== is_duplicate = 1
  Q1: Why does the handlebar of the cyclist's bicycle so low?
  Q2: Why is the handlebar of the cyclist's bicycle so low?

  Q1: What are signs my guy friend is flirting with me?
  Q2: What are signs that your guy friend/ buddy is flirting with you (we are both in our mid twenties)?

  Q1: How can we reduce humidity in our homes? Any tips?
  Q2: How can I reduce humidity in a room?

=== is_duplicate = 0
  Q1: Who would win in a battle between a wolf and a sun bear?
  Q2: Why does all Indian players is with beard in this newzealand test series?

  Q1: When will the academic year starts in IIM?
  Q2: Is IIM and FMS possible with a low grade?

  Q1: What one dead person would you most like to have dinner with, if it were possible?
  Q2: If you could have dinner with one famous person, dead or alive, who would it be, and why?



## Summary
* Task: binary classification of question pairs (duplicate / not).
* ~404k pairs, ~37% duplicates, only a handful of missing questions.
* Next: performance measures, train/test split and basic EDA.